# Connect Lab Notebook data with Inventory items using pyBIS

This guide replicates 1 to 3 steps from the corresponding [GUI tutorial](https://bamresearch.github.io/openBIS-data-store-tutorials/connect-data).

- Step 1: Connect Material Synthesis Experimental Step with chemicals, instrument, and sample from the inventory

- Step 2: Connect Samples from the same Collection

- Step 3: Adding the rest of the parent-child relationship to finalize the tutorial


In [ ]:
from pybis import Openbis
o = Openbis("https://playground.datastore.bam.de/")

o.set_token("your_token")  # Replace with your actual token

## Step 1: Connect Material Synthesis Experimental Step with chemicals and instrument from the inventory

In this step, you will navigate to the Material Synthesis Experimental Step in your Home Space. Next, find needed chemicals, instrument, and sample, save them as variable. Finally, connect these inventory items with the Experimental Step via parent-children relationships. Added relationships in this step are visualized in this hierarchy graph.

![Parent-children relationships of Material Synthesis Experimental Step](../../connect-data/images/step1-connections.png)

Navigate to the your Home Space in the Lab Notebook.

In [ ]:
home_space = o.get_space('X.1_BAMUSERNAME') # Replace with your space code

Find the project with code 'MATERIAL_SYNTHESIS'.

In [ ]:
myproj = next((p for p in home_space.get_projects() if p.attrs.all()['code'] == 'MATERIAL_SYNTHESIS'))
myproj

Then we will find Synthesis Collection.

In [ ]:
mycol = next((c for c in myproj.get_collections() if c.props.all()['$name'] == 'Synthesis'))
mycol.props.all()

Find Material Synthesis Experimental Step. 

In [ ]:
material_synthesis = next((s for s in mycol.get_samples() if s.props.all()['$name'] == 'Material Synthesis'))
material_synthesis.props.all()

Recall parents of the Experimental Step with `get_parents()` function.

In [ ]:
material_synthesis.get_parents()#this should display nothing because the object has no parents yet, we will add a parent in the next step

We need to find the identifiers for chemical 1 and chemical 2, and add the to the `parent_chemicals_identifiers` list.

In [ ]:
materials_space = o.get_space('FB_MATERIALS') #Replace FB with your division number
chemicals_proj = next((p for p in materials_space.get_projects() if p.attrs.all()['code'] == 'CHEMICALS_SYNTHESIS'))
chemicals_collection = next((c for c in chemicals_proj.get_collections() if c.props.all()['$name'] == 'Chemicals'))
parent_chemicals = chemicals_collection.get_samples()
parent_chemicals_identifiers = [s.attrs.all()['identifier'] for s in parent_chemicals if s.props.all()['$name'] in ['Chemical 1', 'Chemical 2']]
parent_chemicals_identifiers

Identifier of the instrument will be added to the `parents_instruments_identifiers` list.

In [ ]:
bam_equipment_space = o.get_space('BAM_EQUIPMENT') 
equipment_open_proj = next((p for p in bam_equipment_space.get_projects() if p.attrs.all()['code'] == 'FB_EQUIPMENT_OPEN')) # Replace FB with your division number
instruments_collection = next((c for c in equipment_open_proj.get_collections() if c.props.all()['$name'] == 'Instruments'))
parents_instruments = instruments_collection.get_samples()
parents_instruments_identifiers = [s.attrs.all()['identifier'] for s in parents_instruments if s.props.all()['$name'] == 'Synthesis']
parents_instruments_identifiers

Add parents to the Material Synthesis Experimental Step using `set_parents()` function.

In [ ]:
material_synthesis.set_parents([parent_chemicals_identifiers[0], parent_chemicals_identifiers[1], parents_instruments_identifiers[0]])#this will add the parents, we can add more parents if we want to, but for now we will just add these three parents
material_synthesis.save()

In [ ]:
material_synthesis.get_parents()#this should display the parents that we just added

Next, let's add children to the Material Synthesis. Find the Synthesized Material Sample that has your username in the co_responsible_person property.

In [ ]:
person_bam = o.get_object(sample_ident="/BAM_GLOBAL/BAM_DATA/BAMUSERNAME") # Replace with your username
person_bam.permId

In [ ]:
#materials_space = o.get_space('X.1_MATERIALS') # Commented out line(s) are reused, and mentioned above, so we don't need to repeat them here
samples_proj = next((p for p in materials_space.get_projects() if p.attrs.all()['code'] == 'SAMPLES_SYNTHESIS'))
samples_collection = next((c for c in samples_proj.get_collections() if c.props.all()['$name'] == 'Samples'))
synthesized_sample = next((s  for s in samples_collection.get_samples() if s.props.all()['$name'] == 'Synthesized Material' and s.props.all()['co_responsible_person'] == person_bam.permId))
synthesized_sample_identifier =  [synthesized_sample.attrs.all()['identifier']]

In [ ]:
# Need to store it again because there are errors that are not explainable
material_synthesis = next((s for s in mycol.get_samples() if s.props.all()['$name'] == 'Material Synthesis'))

In [ ]:
material_synthesis.set_children([synthesized_sample_identifier[0]])
material_synthesis.save()

Now we can check if the child was added correctly by calling the `get_children()`.

In [ ]:
material_synthesis.get_children()

## Step 2: Connect Samples from the same Collection

In this step you will add two children Material Sample 001 and Material Sample 002 to the Synthesized Material. All Sample are from the same Collection. Make sure that the Sample have your username in the co-responsible person property.

Parent-children relationships that will be added in this step are marked in the red box.
![Parent-children relationships that will be added in this step are marked in the red box.](../../connect-data/images/step2-connections.png)


We find and save Material Sample 001 and 002 identifiers for later use.

In [ ]:
#materials_space = o.get_space('FB_MATERIALS') # Commented out line(s) are reused, and mentioned above, so we don't need to repeat them here
#samples_proj = next((p for p in materials_space.get_projects() if p.attrs.all()['code'] == 'SAMPLES_SYNTHESIS'))
#samples_collection = next((c for c in samples_proj.get_collections() if c.props.all()['$name'] == 'Samples'))
ms_001 = next((s for s in samples_collection.get_samples() if s.props.all()['$name'] == 'Material Sample_001' and s.props.all()['co_responsible_person'] == person_bam.permId))
ms_002 = next((s for s in samples_collection.get_samples() if s.props.all()['$name'] == 'Material Sample_002' and s.props.all()['co_responsible_person'] == person_bam.permId))
ms_identifiers = [ms_001.attrs.all()['identifier'], ms_002.attrs.all()['identifier']]
ms_identifiers

Connection Material Sample 001 and Material Sample 002 to the Synthesized Material.

In [ ]:
synthesized_sample.set_children([ms_identifiers[0], ms_identifiers[1]])
synthesized_sample.save()

#or adding parents to the children instead of adding children to the parents.
#ms_001.set_parents([synthesized_sample.identifier])
#ms_001.save()
#ms_002.set_parents([synthesized_sample.identifier])
#ms_002.save()

Two Sample now should be visible in the Synthesized Material.

In [ ]:
synthesized_sample.get_children()

## Step 3: Adding the rest of the parent-child relationship to finalize the tutorial

To finalize the tutorial the rest of the parent-child relationships between the objects need to be added. All parent child relationships that need to be created are highlighted in the red box in the workflow diagram.

We will add the relationship by navigating to each Experimental Step and identifying parents and children. Next, find and save parent and children identifiers as variable. Finally, add the connections to the Experimental Step by implementing the `set_parent()` and `set_children()` function. Save the changes.


![](../connect-data/images/step3-workflow.png)

First, we add the relationship to the Macrostructure Characterization Experimental Step.

Relationships to add to the experimental step **Macrostructure characterization**

| Relationship        | Object type                        | Object Name |
|---------------------|------------------------------------|-------------|
| Parent              | Sample                             |Material Sample 001|
| Parent              | Instrument                         |Macrostructure|

As a result Macrostructure Characterization will have two parents objects, this connection can be viewed in GUI.
![](../connect-data/images/step3-connections-macrostructure-characterization.png)

Find Macrostructure Characterization in the project.

In [ ]:
macrostructure_characterization = next((s for s in myproj.get_samples() if s.props.all()['$name'] == 'Macrostructure Characterization'))
#macrostructure_characterization

Find Macrostructure Instrument, and its identifier.

In [ ]:
#bam_equipment_space = o.get_space('BAM_EQUIPMENT') # Commented out line(s) are reused, and mentioned above, so we don't need to repeat them here
#equipment_open_proj = next((p for p in bam_equipment_space.get_projects() if p.attrs.all()['code'] == 'FB_EQUIPMENT_OPEN')) # Replace FB with your division number
#instruments_collection = next((c for c in equipment_open_proj.get_collections() if c.props.all()['$name'] == 'Instruments'))
macrostructure = next((s for s in instruments_collection.get_samples() if s.props.all()['$name'] == 'Macrostructure'))
macrostructure.identifier

In [ ]:
macrostructure_characterization.set_parents([ms_001.identifier,macrostructure.identifier])
macrostructure_characterization.save()

Next, we will add the parent-children relationship to the Chemical Treatment Experimental Step.

Relationships to add to the experimental step **Chemical Treatment**

|Relationship    |	Object type	    |Object Name|
|---------------|-------------------|-----------|
|Parent	        |Sample             |	Material Sample 002|
|Parent	        |Chemical           |	Treatment Solution|
|Child	        |Sample	            |Treatment Sample 001|
|Child	        |Sample	            |Treatment Sample 002|

Connected Chemical Treatment can be viewed in the GUI.

![](../connect-data/images/step3-connections-chemical-treatment.png)


Find Chemical Treatment in your project.

In [ ]:
chem = next((s for s in myproj.get_samples() if s.props.all()['$name'] == 'Chemical Treatment'))
chem.props.all()

Find Treatment Solution, and save its identifier

In [ ]:
#materials_space = o.get_space('FB_MATERIALS') # Commented out line(s) are reused, and mentioned above, so we don't need to repeat them here
#chemicals_proj = next((p for p in materials_space.get_projects() if p.attrs.all()['code'] == 'CHEMICALS_SYNTHESIS'))
#chemicals_collection = next((c for c in chemicals_proj.get_collections() if c.props.all()['$name'] == 'Chemicals'))
treatment_solution = next((s for s in chemicals_collection.get_samples() if s.props.all()['$name'] == 'Treatment Solution'))
treatment_solution.identifier

Find Treated Sample 001 and Treated Sample 002, save their identifiers.

In [ ]:
#materials_space = o.get_space('FB_MATERIALS') # Commented out line(s) are reused, and mentioned above, so we don't need to repeat them here
#samples_proj = next((p for p in materials_space.get_projects() if p.attrs.all()['code'] == 'SAMPLES_SYNTHESIS'))
#samples_collection = next((c for c in samples_proj.get_collections() if c.props.all()['$name'] == 'Samples'))
tr_001 = next((s for s in samples_collection.get_samples() if s.props.all()['$name'] == 'Treated Sample_001' and s.props.all()['co_responsible_person'] == person_bam.permId))
tr_002 = next((s for s in samples_collection.get_samples() if s.props.all()['$name'] == 'Treated Sample_002' and s.props.all()['co_responsible_person'] == person_bam.permId))
print(tr_001.identifier, tr_002.identifier)

Add parents and children to the Chemical Treatment.

In [ ]:
chem.set_parents([ms_002.identifier, treatment_solution.identifier])
chem.set_children([tr_001.identifier, tr_002.identifier])
chem.save()

Relationships to add to the experimental step **Macrostructure characterization TS**

|Relationship	|Object type	|Object Name|
|---------------|---------------|-----------|
|Parent	        |Sample	        |Treated Sample 001|	
|Parent	        |Instrument	    |Macrostructure|

After the steps below following result should be seen in GUI.

![](../connect-data/images/step3-connections-macrostructure-characterization-ts.png)

Find Macrostructure characterization TS in your project.

In [ ]:
macro_ts = next((s for s in myproj.get_samples() if s.props.all()['$name'] == 'Macrostructure Characterization TS'))
macro_ts.props.all()

Add both parent, which are already saved as variables `tr_001` for Treated Sample 001 and `macrostructure` for Macrostructure Instrument.

In [ ]:
macro_ts.set_parents([tr_001.identifier, macrostructure.identifier])
macro_ts.save()

Then we connect the Microstructure Characterization

Relationships to add to the experimental step "Microstructure characterization"
|Relationship	|Object type	    |Object Name	    |
|---|--|--|
|Parent	        |Sample	            |Treated Sample 002	|
|Parent	        |Instrument	        |Microstructure	|
|Child	        |Experimental Step	|Microstructure Data Analysis	    |

As a result you will see connections in the GUI.

![](../connect-data/images/step3-connections-microstructure-characterization.png)


In [ ]:
micro = next((s for s in myproj.get_samples() if s.props.all()['$name'] == 'Microstructure Characterization'))
micro.props.all()

We already have Treated Sample_002 saved in tr_002 variable so we reuse it. We need to find Microstructure Data Analysis and Microstructure Instrument, and save their identifiers to proceed further.


In [ ]:
micro_da = next((s for s in myproj.get_samples() if s.props.all()['$name'] == 'Microstructure data analysis'))
micro_da.identifier

In [ ]:
#bam_equipment_space = o.get_space('BAM_EQUIPMENT')  # Commented out line(s) are reused, and mentioned above, so we don't need to repeat them here
#equipment_open_proj = next((p for p in bam_equipment_space.get_projects() if p.attrs.all()['code'] == 'X.1_EQUIPMENT_OPEN'))
#instruments_collection = next((c for c in equipment_open_proj.get_collections() if c.props.all()['$name'] == 'Instruments'))
microstructure = next((s for s in instruments_collection.get_samples() if s.props.all()['$name'] == 'Microstructure'))
microstructure.identifier

Now we are ready to add parents and children to Microstructure Characterization.

In [ ]:
micro.set_parents([tr_002.identifier, microstructure.identifier])
micro.set_children([micro_da.identifier])
micro.save()

Observe the result in GUI by navigating to the Experimental Step, clicking on **More → Hierarchy Graph**.